# Experiment 9 — Scratch-Trained Residual CNN

**Role:** Person 1 — baseline model  
**Task:** Classify 200 bird species using a CNN trained from random initialization.

This notebook documents the final Experiment 9 setup, training procedure, and evaluation. It is organized for reproducibility and does not automatically retrain the model. To reproduce training, set `RUN_TRAINING = True` in the training cell.

**Recorded results from the completed run**
- Best validation accuracy: **53.39%** (epoch 91)
- Standard test accuracy: **56.04%**
- Test accuracy with horizontal-flip test-time augmentation (TTA): **56.56%**

The results above are recorded from the completed experiment. They will be recalculated only when the relevant evaluation cells are run with the saved checkpoint and data available.

## 1. Imports and configuration

This notebook expects the project split file at `outputs/eda/splits.csv` and the CUB train/test datasets to be accessible through Hugging Face `datasets`.

In [ ]:
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Find project directory without assuming the notebook's current directory.
candidate_dirs = [
    Path("/kaggle/working/Deep-project"),
    Path.cwd(),
]
PROJECT_DIR = next(
    (p for p in candidate_dirs if (p / "outputs/eda/splits.csv").exists()),
    None
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find outputs/eda/splits.csv. Clone/open the Deep-project "
        "repository or set PROJECT_DIR to the repository root."
    )

SPLITS_PATH = PROJECT_DIR / "outputs/eda/splits.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs/baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Splits file:", SPLITS_PATH)
print("Output directory:", OUTPUT_DIR)

## 2. Load the dataset splits

The project CSV uses the `partition` column with `train`, `validation`, and `test` values. The validation images are drawn from the training source; the test images use the separate test source.

In [ ]:
splits = pd.read_csv(SPLITS_PATH)

required_columns = {"partition", "index", "file_name", "label"}
missing_columns = required_columns - set(splits.columns)
if missing_columns:
    raise ValueError(f"Missing split CSV columns: {sorted(missing_columns)}")

train_rows = splits[splits["partition"] == "train"].copy().reset_index(drop=True)
val_rows = splits[splits["partition"] == "validation"].copy().reset_index(drop=True)
test_rows = splits[splits["partition"] == "test"].copy().reset_index(drop=True)

print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Test:", len(test_rows))

assert len(train_rows) == 5092
assert len(val_rows) == 899
assert len(test_rows) == 5794

train_source = load_dataset("Multimodal-Fatima/CUB_train", split="train")
test_source = load_dataset("Multimodal-Fatima/CUB_test", split="test")

print("Train source size:", len(train_source))
print("Test source size:", len(test_source))

## 3. Image preprocessing and data loaders

Images are cached at 144 × 144 pixels, then augmented and transformed to 128 × 128 tensors during training. Validation and test use the same deterministic center crop and normalization.

In [ ]:
MEAN = [0.484575231096779, 0.4990690590550937, 0.4326982401975864]
STD = [0.2320364540940151, 0.22752646981271832, 0.2660812817755806]

cache_transform = T.Compose([
    T.Resize(144),
    T.CenterCrop(144),
])

normalization = T.Normalize(mean=MEAN, std=STD)

train_transform = T.Compose([
    T.RandomResizedCrop(128, scale=(0.35, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.1),
    T.ToTensor(),
    normalization,
    T.RandomErasing(p=0.25),
])

eval_transform = T.Compose([
    T.CenterCrop(128),
    T.ToTensor(),
    normalization,
])

def to_pil(image):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    return image.convert("RGB")

def cache_images(rows, source, description):
    images, labels = [], []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=description):
        record = source[int(row["index"])]
        images.append(cache_transform(to_pil(record["image"])))
        labels.append(int(row["label"]))
    return images, labels

class CachedBirdDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return self.transform(self.images[index]), self.labels[index]

train_images, train_labels = cache_images(train_rows, train_source, "Caching train images")
val_images, val_labels = cache_images(val_rows, train_source, "Caching validation images")
test_images, test_labels = cache_images(test_rows, test_source, "Caching test images")

train_dataset = CachedBirdDataset(train_images, train_labels, train_transform)
val_dataset = CachedBirdDataset(val_images, val_labels, eval_transform)
test_dataset = CachedBirdDataset(test_images, test_labels, eval_transform)

# num_workers=0 is deliberate for better compatibility in notebook environments.
train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=0,
    drop_last=True, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=64, shuffle=False, num_workers=0,
    pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, num_workers=0,
    pin_memory=(device.type == "cuda")
)

print(f"Loader sizes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

## 4. Model architecture

The model is imported from `src/models/scratch_resnet.py`, the canonical implementation used by the Python training/evaluation script and the saved Experiment 9 checkpoint. It is a residual CNN initialized from scratch; no pretrained weights are loaded.


In [ ]:
# Import the canonical architecture from src/models/scratch_resnet.py.
# Keeping one model definition ensures the notebook matches best.pth and the CLI script.
import sys
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))
from models.scratch_resnet import ScratchResNet

model = ScratchResNet(num_classes=200).to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {parameter_count:,}")


## 5. Training configuration and reproducible training function

The completed run used AdamW, label smoothing, CutMix, mixed precision on CUDA, and OneCycleLR for 100 epochs. Training is **disabled by default** so running the notebook does not accidentally start another 100-epoch job.

To reproduce training from scratch, set `RUN_TRAINING = True` in the next cell. This will write a new checkpoint and history to `outputs/baseline/`.

In [ ]:
EPOCHS = 100
MAX_LR = 2e-3
WEIGHT_DECAY = 0.03
LABEL_SMOOTHING = 0.1
RUN_TRAINING = False  # Change to True only when intentionally retraining.

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def cutmix(images, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    permutation = torch.randperm(images.size(0), device=images.device)
    height, width = images.shape[2:]

    cut_ratio = np.sqrt(1.0 - lam)
    cut_h, cut_w = int(height * cut_ratio), int(width * cut_ratio)
    center_y, center_x = np.random.randint(height), np.random.randint(width)

    y1 = max(center_y - cut_h // 2, 0)
    y2 = min(center_y + cut_h // 2, height)
    x1 = max(center_x - cut_w // 2, 0)
    x2 = min(center_x + cut_w // 2, width)

    mixed_images = images.clone()
    mixed_images[:, :, y1:y2, x1:x2] = images[permutation, :, y1:y2, x1:x2]
    lam = 1.0 - ((y2 - y1) * (x2 - x1) / (height * width))
    return mixed_images, labels, labels[permutation], lam

@torch.no_grad()
def evaluate(model, loader, tta=False):
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0
    use_amp = device.type == "cuda"

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            if tta:
                flipped_logits = model(torch.flip(images, dims=[3]))
                logits = (logits + flipped_logits) / 2
            loss = nn.functional.cross_entropy(logits, labels)

        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return loss_sum / total, correct / total

def train_model(model, train_loader, val_loader):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=MAX_LR, epochs=EPOCHS,
        steps_per_epoch=len(train_loader), pct_start=0.1
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best_val_acc = 0.0
    history = []
    checkpoint_path = OUTPUT_DIR / "best.pth"
    history_path = OUTPUT_DIR / "history.csv"

    for epoch in range(EPOCHS):
        model.train()
        loss_sum = 0.0
        correct = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, enabled=use_amp):
                if np.random.rand() < 0.5:
                    mixed_images, labels_a, labels_b, lam = cutmix(images, labels)
                    logits = model(mixed_images)
                    loss = lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)
                else:
                    logits = model(images)
                    loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            batch_size = labels.size(0)
            loss_sum += loss.item() * batch_size
            correct += (logits.argmax(1) == labels).sum().item()
            total += batch_size

        val_loss, val_acc = evaluate(model, val_loader)
        record = {
            "epoch": epoch + 1,
            "train_loss": loss_sum / total,
            "train_acc": correct / total,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(history_path, index=False)

        print(
            f"Epoch {epoch + 1:03d} | "
            f"train loss {record['train_loss']:.3f} | "
            f"val loss {val_loss:.3f} | val acc {val_acc * 100:.2f}%"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "best_val_accuracy": val_acc,
            }, checkpoint_path)

    return pd.DataFrame(history)

if RUN_TRAINING:
    history = train_model(model, train_loader, val_loader)
else:
    print("Training skipped. Use the saved checkpoint/history for final reporting.")

## 6. Training curves

This section loads the saved history file. Place the completed run's `history.csv` in `outputs/baseline/history.csv` before running this section.

In [ ]:
history_path = OUTPUT_DIR / "history.csv"

if history_path.exists():
    history = pd.read_csv(history_path)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["epoch"], history["train_loss"], label="Training loss")
    ax.plot(history["epoch"], history["val_loss"], label="Validation loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Experiment 9 — Loss")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "loss_curves.png", dpi=200)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["epoch"], history["train_acc"] * 100, label="Training accuracy")
    ax.plot(history["epoch"], history["val_acc"] * 100, label="Validation accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Experiment 9 — Accuracy")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "accuracy_curves.png", dpi=200)
    plt.show()

    best_row = history.loc[history["val_acc"].idxmax()]
    print(f"Best validation accuracy: {best_row['val_acc'] * 100:.2f}% at epoch {int(best_row['epoch'])}")
else:
    print(f"No training history found at {history_path}. Copy the completed run's history.csv here.")

## 7. Load the best checkpoint and evaluate on the test set

This section does not train the model. It loads the saved best checkpoint and reports standard test performance, then separately reports horizontal-flip TTA performance.

Before running, ensure `best.pth` from the completed Experiment 9 is available at `outputs/baseline/best.pth`.

In [ ]:
checkpoint_path = OUTPUT_DIR / "best.pth"

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {checkpoint_path}. Copy the completed Experiment 9 best.pth "
        "to outputs/baseline/best.pth, or update checkpoint_path to its current location."
    )

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
test_model = ScratchResNet(num_classes=200).to(device)
test_model.load_state_dict(checkpoint["model_state_dict"])
test_model.eval()

print("Checkpoint epoch:", checkpoint.get("epoch"))
print(f"Saved best validation accuracy: {checkpoint.get('best_val_accuracy', float('nan')) * 100:.2f}%")

test_loss, test_acc = evaluate(test_model, test_loader, tta=False)
print(f"Standard test loss: {test_loss:.4f}")
print(f"Standard test accuracy: {test_acc * 100:.2f}%")

test_loss_tta, test_acc_tta = evaluate(test_model, test_loader, tta=True)
print(f"Test loss with flip TTA: {test_loss_tta:.4f}")
print(f"Test accuracy with flip TTA: {test_acc_tta * 100:.2f}%")

## 8. Results summary

Recorded results from the completed Experiment 9 run:

| Metric | Result |
|---|---:|
| Best validation accuracy | 53.39% |
| Best checkpoint epoch | 91 |
| Standard test accuracy | 56.04% |
| Test accuracy with horizontal-flip TTA | 56.56% |
| Number of classes | 200 |
| Training images | 5,092 |
| Validation images | 899 |
| Test images | 5,794 |

**Conclusion:** The residual CNN was trained from random initialization and achieved 53.39% best validation accuracy and 56.04% standard test accuracy. Horizontal-flip TTA increased the reported test accuracy to 56.56%. TTA results are reported separately because each test image is evaluated in two orientations.